# 2 — Entraînement (transfer learning) & Évaluation

| Étape | Données | Sortie |
|-------|---------|--------|
| Pré-entraînement | cohorte **pédiatrique** | `models/pretrained.keras` |
| **Fine-tuning** (gel de `feat_1`) | cohorte **clinique** | `models/finetuned.keras` |
| Évaluation + SHAP + drift | test clinique | `exports/*.csv`, `reports/*.png` |

Le modèle final `finetuned.keras` est celui utilisé par l'application
(`src/predict.py`, `src/care_agent.py`).

> Prérequis : exécuter d'abord **`1_data_preparation.ipynb`**.

In [ ]:
import os, sys
from pathlib import Path

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import joblib
import keras
import numpy as np
import pandas as pd
from keras import layers

SRC = Path.cwd().parent / "src" if Path.cwd().name == "notebooks" else Path.cwd() / "src"
sys.path.append(str(SRC))
import config as C

keras.utils.set_random_seed(C.RANDOM_SEED)
data = joblib.load(C.PROCESSED_DIR / "datasets.joblib")
n_features = len(data["feature_names"])
print("Features :", data["feature_names"])

## Architecture du réseau (MLP tabulaire)

Les couches sont **nommées** pour permettre le gel sélectif au fine-tuning :
on gèle `feat_1` et on ré-entraîne `feat_2` + `head`.

In [ ]:
def compile_model(model, lr):
    model.compile(optimizer=keras.optimizers.Adam(lr), loss="binary_crossentropy",
                  metrics=[keras.metrics.AUC(name="auc"), "accuracy",
                           keras.metrics.Precision(name="precision"),
                           keras.metrics.Recall(name="recall")])
    return model


def build_mlp(input_dim, lr=1e-3):
    model = keras.Sequential([
        layers.Input((input_dim,), name="input"),
        layers.Dense(32, activation="relu", name="feat_1"),
        layers.Dropout(0.30, name="drop_1"),
        layers.Dense(16, activation="relu", name="feat_2"),
        layers.Dropout(0.20, name="drop_2"),
        layers.Dense(1, activation="sigmoid", name="head"),
    ], name="anemia_mlp")
    return compile_model(model, lr)


def recompile_for_finetuning(model, freeze="feat_1", lr=2e-4):
    for layer in model.layers:
        layer.trainable = layer.name != freeze
    return compile_model(model, lr)


def class_weights(y):
    y = np.asarray(y).astype(int); n = len(y)
    return {c: (n / (2.0 * max((y == c).sum(), 1))) for c in (0, 1)}


def callbacks(monitor="val_auc"):
    return [keras.callbacks.EarlyStopping(monitor=monitor, mode="max", patience=25,
                                          restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(monitor=monitor, mode="max", factor=0.5,
                                              patience=10, min_lr=1e-5)]

## Étape 1 — Pré-entraînement (cohorte pédiatrique)

In [ ]:
pre = build_mlp(n_features, lr=1e-3)
hist_pre = pre.fit(data["X_pre_train"], data["y_pre_train"],
                   validation_data=(data["X_pre_val"], data["y_pre_val"]),
                   epochs=300, batch_size=32, class_weight=class_weights(data["y_pre_train"]),
                   callbacks=callbacks("val_auc"), verbose=0)
pre.save(C.MODELS_DIR / "pretrained.keras")

hdf = pd.DataFrame(hist_pre.history); hdf.insert(0, "epoch", range(1, len(hdf) + 1))
hdf.insert(0, "phase", "pré-entraînement")
hdf.to_csv(C.EXPORTS_DIR / "history_pretrain.csv", index=False)
val = pre.evaluate(data["X_pre_val"], data["y_pre_val"], verbose=0, return_dict=True)
print(f"[Pré-entraînement] époques={len(hdf)} val_auc={val['auc']:.3f} val_acc={val['accuracy']:.3f}")

## Étape 2 — Fine-tuning (cohorte clinique)

In [ ]:
from sklearn.metrics import f1_score, roc_auc_score


def load_finetunable():
    net = keras.models.load_model(C.MODELS_DIR / "pretrained.keras")
    return recompile_for_finetuning(net, freeze="feat_1", lr=2e-4)


def fit_eval(net, Xtr, ytr, seed=C.RANDOM_SEED):
    keras.utils.set_random_seed(seed)
    h = net.fit(Xtr, ytr, validation_data=(data["X_clin_val"], data["y_clin_val"]),
                epochs=300, batch_size=32, class_weight=class_weights(ytr),
                callbacks=callbacks("val_auc"), verbose=0)
    p = net.predict(data["X_clin_test"], verbose=0).ravel()
    return net, h, roc_auc_score(data["y_clin_test"], p), f1_score(data["y_clin_test"], (p >= 0.5).astype(int))


ft, h_ft, auc_ft, f1_ft = fit_eval(load_finetunable(), data["X_clin_train"], data["y_clin_train"])
ft.save(C.MODELS_DIR / "finetuned.keras")
hdf = pd.DataFrame(h_ft.history); hdf.insert(0, "epoch", range(1, len(hdf) + 1))
hdf.insert(0, "phase", "fine-tuning")
hdf.to_csv(C.EXPORTS_DIR / "history_finetune.csv", index=False)
print(f"[Fine-tuning] AUC_test={auc_ft:.3f} F1_test={f1_ft:.3f} (époques={len(hdf)})")
print("Modèle final ->", C.MODELS_DIR / "finetuned.keras")

## Efficacité-données : transfert vs *from scratch*

Le bénéfice du transfert est maximal quand les données cliniques sont rares.

In [ ]:
import matplotlib.pyplot as plt


def subsample(X, y, frac, seed=C.RANDOM_SEED):
    if frac >= 1.0:
        return X, y
    rng = np.random.default_rng(seed); idx = []
    for c in (0, 1):
        ci = np.where(y == c)[0]; k = max(2, int(round(len(ci) * frac)))
        idx.extend(rng.choice(ci, size=min(k, len(ci)), replace=False))
    idx = np.array(idx); return X[idx], y[idx]


rows = []
for fr in [0.05, 0.10, 0.25, 0.50, 1.00]:
    Xtr, ytr = subsample(data["X_clin_train"], data["y_clin_train"], fr)
    _, _, a_scr, _ = fit_eval(build_mlp(n_features, 1e-3), Xtr, ytr)
    _, _, a_tr, _ = fit_eval(load_finetunable(), Xtr, ytr)
    rows.append({"fraction": fr, "n_train": len(ytr), "auc_scratch": round(a_scr, 4),
                 "auc_transfer": round(a_tr, 4), "gain_transfer": round(a_tr - a_scr, 4)})
    print(f"{fr:>5.0%} n={len(ytr):>4d} scratch={a_scr:.3f} transfer={a_tr:.3f} gain={a_tr - a_scr:+.3f}")

lift = pd.DataFrame(rows); lift.to_csv(C.EXPORTS_DIR / "transfer_lift.csv", index=False)
plt.figure(figsize=(6, 4))
plt.plot(lift["fraction"], lift["auc_scratch"], "o--", label="from scratch")
plt.plot(lift["fraction"], lift["auc_transfer"], "o-", label="transfer learning")
plt.xlabel("Fraction du train clinique"); plt.ylabel("AUC (test)"); plt.legend()
plt.title("Efficacité-données"); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## Évaluation du modèle final + explicabilité

Métriques de classification, matrice de confusion, ROC/PR, calibration,
importance SHAP et drift PSI (pédiatrique → clinique).

In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import (average_precision_score, brier_score_loss, confusion_matrix,
                             log_loss, precision_recall_curve, roc_curve)

model = keras.models.load_model(C.MODELS_DIR / "finetuned.keras")
Xte, yte = data["X_clin_test"], data["y_clin_test"].astype(int)
proba = model.predict(Xte, verbose=0).ravel(); pred = (proba >= 0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(yte, pred).ravel()

metrics = {
    "n_test": int(len(yte)), "prevalence": float(yte.mean()),
    "accuracy": float((pred == yte).mean()),
    "precision": float(tp / (tp + fp)) if (tp + fp) else 0.0,
    "recall_sensibilite": float(tp / (tp + fn)) if (tp + fn) else 0.0,
    "specificite": float(tn / (tn + fp)) if (tn + fp) else 0.0,
    "f1": float(f1_score(yte, pred)), "roc_auc": float(roc_auc_score(yte, proba)),
    "pr_auc": float(average_precision_score(yte, proba)),
    "brier_score": float(brier_score_loss(yte, proba)),
    "log_loss": float(log_loss(yte, np.clip(proba, 1e-6, 1 - 1e-6))),
}
pd.DataFrame({"metrique": list(metrics), "valeur": list(metrics.values())}).to_csv(
    C.EXPORTS_DIR / "model_metrics.csv", index=False)

pd.DataFrame([["Non anémique", "Non anémique", int(tn)], ["Non anémique", "Anémique", int(fp)],
              ["Anémique", "Non anémique", int(fn)], ["Anémique", "Anémique", int(tp)]],
             columns=["reel", "predit", "n"]).to_csv(C.EXPORTS_DIR / "confusion_matrix.csv", index=False)

fpr, tpr, thr = roc_curve(yte, proba)
pd.DataFrame({"fpr": fpr, "tpr": tpr, "seuil": np.clip(thr, 0, 1)}).to_csv(
    C.EXPORTS_DIR / "roc_curve.csv", index=False)
prec, rec, _ = precision_recall_curve(yte, proba)
pd.DataFrame({"recall": rec, "precision": prec}).to_csv(C.EXPORTS_DIR / "pr_curve.csv", index=False)
frac_pos, mean_pred = calibration_curve(yte, proba, n_bins=10, strategy="quantile")
pd.DataFrame({"proba_moyenne_predite": mean_pred, "frequence_observee": frac_pos}).to_csv(
    C.EXPORTS_DIR / "calibration_curve.csv", index=False)
print("=== Métriques IA (test) ===")
for k, v in metrics.items():
    print(f"   {k:>18}: {v:.4f}" if isinstance(v, float) else f"   {k:>18}: {v}")

In [ ]:
# Importance SHAP (repli sur permutation) + drift PSI
def shap_importance(model, data, feats):
    f = lambda x: model.predict(x, verbose=0).ravel()
    Xtr, Xte = data["X_clin_train"], data["X_clin_test"]
    try:
        import shap
        rng = np.random.default_rng(C.RANDOM_SEED)
        bg = Xtr[rng.choice(len(Xtr), min(100, len(Xtr)), replace=False)]
        sample = Xte[rng.choice(len(Xte), min(150, len(Xte)), replace=False)]
        sv = shap.Explainer(f, shap.maskers.Independent(bg))(sample)
        imp, method = np.abs(sv.values).mean(axis=0), "shap"
    except Exception as e:
        print("   [SHAP -> permutation]", e)
        base = f(Xte); base_err = np.mean((base - data["y_clin_test"]) ** 2)
        rng = np.random.default_rng(C.RANDOM_SEED); imp = []
        for j in range(Xte.shape[1]):
            Xp = Xte.copy(); Xp[:, j] = rng.permutation(Xp[:, j])
            imp.append(np.mean((f(Xp) - data["y_clin_test"]) ** 2) - base_err)
        imp, method = np.array(imp), "permutation"
    return pd.DataFrame({"feature": feats, "importance_moyenne": imp, "methode": method}
                        ).sort_values("importance_moyenne", ascending=False).reset_index(drop=True)


def psi(expected, actual, bins=10):
    q = np.quantile(expected, np.linspace(0, 1, bins + 1)); q[0], q[-1] = -np.inf, np.inf
    q = np.unique(q)
    e = np.clip(np.histogram(expected, q)[0] / len(expected), 1e-4, None)
    a = np.clip(np.histogram(actual, q)[0] / len(actual), 1e-4, None)
    return float(np.sum((a - e) * np.log(a / e)))


shap_imp = shap_importance(model, data, data["feature_names"])
shap_imp.to_csv(C.EXPORTS_DIR / "shap_importance.csv", index=False)

ped = pd.read_csv(C.PROCESSED_DIR / "pediatric_clean.csv")
clin = pd.read_csv(C.PROCESSED_DIR / "clinical_clean.csv")
drift = pd.DataFrame([
    {"feature": f, "psi": round(psi(ped[f].to_numpy(), clin[f].to_numpy()), 4),
     "interpretation": "stable" if psi(ped[f].to_numpy(), clin[f].to_numpy()) < 0.1
     else "décalage modéré" if psi(ped[f].to_numpy(), clin[f].to_numpy()) < 0.25 else "décalage majeur"}
    for f in C.CONTINUOUS_FEATURES]).sort_values("psi", ascending=False)
drift.to_csv(C.EXPORTS_DIR / "feature_drift.csv", index=False)
print("Top features (SHAP) :", list(shap_imp["feature"].head(3)))
shap_imp

In [ ]:
# Figures pour le rapport
plt.figure(figsize=(5, 4)); plt.plot(fpr, tpr, label=f"AUC={metrics['roc_auc']:.3f}")
plt.plot([0, 1], [0, 1], "--", color="grey"); plt.xlabel("FPR"); plt.ylabel("Sensibilité")
plt.title("Courbe ROC"); plt.legend(); plt.tight_layout()
plt.savefig(C.REPORTS_DIR / "roc.png", dpi=120); plt.show()

plt.figure(figsize=(5, 4)); plt.plot(mean_pred, frac_pos, "o-", label="modèle")
plt.plot([0, 1], [0, 1], "--", color="grey", label="parfaite"); plt.legend()
plt.title(f"Calibration (Brier={metrics['brier_score']:.3f})"); plt.tight_layout()
plt.savefig(C.REPORTS_DIR / "calibration.png", dpi=120); plt.show()

s = shap_imp.sort_values("importance_moyenne")
plt.figure(figsize=(5, 4)); plt.barh(s["feature"], s["importance_moyenne"], color="#2c7fb8")
plt.title("Importance des features (SHAP)"); plt.tight_layout()
plt.savefig(C.REPORTS_DIR / "shap_importance.png", dpi=120); plt.show()
print("Évaluation terminée -> exports/ + reports/")

## Suite

1. `python src/care_agent.py --bootstrap` → peuple PostgreSQL depuis le modèle.
2. `python src/build_pbip.py` → tableau de bord Power BI branché sur la base.